# 02 â€” Policy Evaluation

**Goal:** Evaluate how threshold choice affects the precision/recall trade-off
across all three configurations, identify the optimal operating point,
and visualise PR and ROC curves.

## Overview
- Section 1: Setup
- Section 2: Threshold sweep for Config B and C
- Section 3: Precision-Recall curves
- Section 4: ROC curves
- Section 5: F1-optimal threshold selection
- Section 6: Strategy comparison (block / annotate / redact)

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))
import warnings; warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import numpy as np

from prompt_injection.detector import InjectionDetector, LogisticRegressionScorer
from prompt_injection.policy import PolicyEngine, PolicyDecision
from prompt_injection.evaluation.dataset import SyntheticDataset
from prompt_injection.evaluation.metrics import compute_metrics, threshold_sweep

SEED = 42; FIGSIZE = (13, 5)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

ds = SyntheticDataset(n_injections=250, n_benign=250, seed=SEED).generate()
train_ds, test_ds = ds.train_test_split(test_size=0.20, seed=SEED)
test_records = test_ds.records()
test_labels  = test_ds.labels()
test_texts   = test_ds.texts()

clf = LogisticRegressionScorer().fit(train_ds.texts(), train_ds.labels())
detectors = {
    'B: Regex + Scoring': InjectionDetector(mode='hybrid', threshold=0.50),
    'C: + Classifier':    InjectionDetector(mode='full',   threshold=0.50, classifier=clf),
}
print("Setup complete.")

## 2. Threshold Sweep

In [ ]:
sweeps = {}
for name, det in detectors.items():
    scores = [det.scan(t).risk_score for t in test_texts]
    sweeps[name] = {'scores': scores, 'sweep': threshold_sweep(test_labels, scores, n_thresholds=200)}

print("Sweep complete.")
for name, data in sweeps.items():
    best = max(data['sweep'], key=lambda p: p.f1)
    print(f"  {name}: best F1={best.f1:.4f} at threshold={best.threshold:.3f}  "
          f"(P={best.precision:.3f}, R={best.recall:.3f})")

## 3. Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
colors = {'B: Regex + Scoring': '#e67e22', 'C: + Classifier': '#8e44ad'}

for ax, (name, data) in zip(axes, sweeps.items()):
    pts = data['sweep']
    recalls    = [p.recall    for p in pts]
    precisions = [p.precision for p in pts]
    f1s        = [p.f1        for p in pts]

    ax.plot(recalls, precisions, color=colors[name], linewidth=2)
    best = max(pts, key=lambda p: p.f1)
    ax.scatter([best.recall], [best.precision], color='red', s=80, zorder=5,
               label=f'Best F1={best.f1:.3f} (thr={best.threshold:.2f})')
    ax.axvline(0.80, color='grey', linestyle=':', alpha=0.6, label='Recall=0.80')
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_xlim(0, 1.05); ax.set_ylim(0, 1.05)
    ax.set_title(name, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle("Precision-Recall Curves", fontweight='bold')
plt.tight_layout()
plt.savefig("pr_curves.png", dpi=130, bbox_inches='tight')
plt.show()

## 4. ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
for ax, (name, data) in zip(axes, sweeps.items()):
    pts  = data['sweep']
    fprs = [p.fpr    for p in pts]
    tprs = [p.recall for p in pts]
    from prompt_injection.evaluation.metrics import _roc_auc
    auc  = _roc_auc(test_labels, data['scores'])

    ax.plot(fprs, tprs, color=colors[name], linewidth=2, label=f'AUC={auc:.4f}')
    ax.plot([0,1], [0,1], 'k--', linewidth=1, alpha=0.4, label='Random')
    ax.fill_between(fprs, tprs, alpha=0.08, color=colors[name])
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
    ax.set_title(name, fontweight='bold')
    ax.legend(fontsize=10)

plt.suptitle("ROC Curves", fontweight='bold')
plt.tight_layout()
plt.savefig("roc_curves.png", dpi=130, bbox_inches='tight')
plt.show()

## 5. F1-Optimal Threshold Selection

In [ ]:
print(f"{'Config':<28} {'Opt Threshold':>14} {'F1':>7} {'Precision':>10} {'Recall':>8}")
print("-" * 72)
for name, data in sweeps.items():
    pts  = data['sweep']
    best = max(pts, key=lambda p: p.f1)
    print(f"{name:<28} {best.threshold:>14.4f} {best.f1:>7.4f} {best.precision:>10.4f} {best.recall:>8.4f}")
print()
print("Recommendation: use the F1-optimal threshold as the default.")
print("For high-stakes deployment, prefer a threshold that maximises recall")
print("(lower threshold) and accept lower precision â€” false negatives are")
print("more dangerous than false positives in prompt-injection defence.")

## 6. Strategy Comparison â€” Block vs Annotate

In [ ]:
det_b = InjectionDetector(mode='hybrid', threshold=0.50)
strategies = ['allow', 'annotate', 'block']
strategy_stats = {}

for strat in strategies:
    engine = PolicyEngine(strategy=strat, threshold=0.50)
    blocked = allowed = annotated = 0
    for record in test_records:
        dr  = det_b.scan(record.text, source_type=record.source_type or 'user')
        pol = engine.decide(dr, record.text)
        if pol.action == PolicyDecision.BLOCK:    blocked   += 1
        elif pol.action == PolicyDecision.ANNOTATE: annotated += 1
        else:                                     allowed   += 1
    strategy_stats[strat] = {'blocked': blocked, 'annotated': annotated, 'allowed': allowed}

print(f"{'Strategy':<12} {'Blocked':>8} {'Annotated':>10} {'Allowed':>8}")
print("-" * 42)
for strat, s in strategy_stats.items():
    print(f"{strat:<12} {s['blocked']:>8} {s['annotated']:>10} {s['allowed']:>8}")
print()
print("'block'   â†’ strongest protection, highest user friction on false positives")
print("'annotate'â†’ zero friction, all detections logged for review")
print("'redact'  â†’ middle ground: suspicious spans replaced with placeholder")